# 05. 파이썬 기초 - 웹 요청과 HTML 파싱

웹스크래핑의 기본 흐름은 **① 요청(requests) → ② 응답 받기 → ③ 파싱(BeautifulSoup) → ④ 표로 정리(pandas)** 입니다.
앞의 1~4편에서 배운 자료구조·함수·예외처리·pandas 가 모두 여기서 쓰입니다.

**다루는 내용**
1. requests 로 웹 요청 보내기
2. 응답 상태 확인과 예외처리
3. JSON API 응답 다루기
4. BeautifulSoup 로 HTML 파싱
5. 파싱 결과 → DataFrame

> 실행하려면: `pip install requests beautifulsoup4`

## 1. requests 로 웹 요청 보내기

`requests.get(url)` 으로 웹페이지/데이터를 받아옵니다.
연습용 사이트 **httpbin.org** 로 안전하게 테스트합니다.

In [ ]:
import requests

# GET 요청 보내기
res = requests.get('https://httpbin.org/get')

print('상태 코드:', res.status_code)   # 200 이면 성공
print('응답 형식:', res.headers.get('Content-Type'))

# 쿼리 파라미터(검색어 등) 함께 보내기 → URL 뒤에 ?key=value 로 붙는다
params = {'query': '파이썬', 'display': 10}
res2 = requests.get('https://httpbin.org/get', params=params)
print('\n실제 요청된 URL:', res2.url)

## 2. 응답 상태 확인과 예외처리

네트워크 요청은 실패할 수 있으므로 03편의 `try/except` 로 감쌉니다.
`raise_for_status()` 는 200이 아니면 예외를 발생시켜 오류를 조기에 잡습니다.

In [ ]:
def fetch(url):
    """URL 에 GET 요청을 보내고 응답 객체를 반환한다. 실패 시 None."""
    try:
        res = requests.get(url, timeout=10)   # timeout: 10초 넘으면 포기
        res.raise_for_status()                # 200 아니면 예외 발생
        return res
    except requests.exceptions.RequestException as e:
        print(f'요청 실패: {e}')
        return None

# 정상 요청
print('성공:', fetch('https://httpbin.org/status/200') is not None)
# 일부러 404 (없는 페이지)
print('실패 처리:', fetch('https://httpbin.org/status/404'))

## 3. JSON API 응답 다루기

많은 API 는 JSON 을 돌려줍니다. `res.json()` 으로 파이썬 딕셔너리로 변환합니다.
01편의 중첩 구조 지식이 그대로 쓰입니다.

In [ ]:
# httpbin 은 보낸 데이터를 JSON 으로 되돌려준다
res = requests.get('https://httpbin.org/get', params={'q': 'kpop', 'page': 1})
data = res.json()          # JSON → 딕셔너리

print('타입:', type(data))
# 딕셔너리 → 키로 접근 (중첩 구조 탐색)
print('보낸 파라미터:', data['args'])
print('q 값:', data['args']['q'])

# 실제 네이버 API 코드의 res.json().get('items', []) 와 같은 방식으로
# 없을 수도 있는 키는 .get(키, 기본값) 으로 안전하게 꺼냅니다.
print('없는 키 안전 조회:', data.get('items', []))

## 4. BeautifulSoup 로 HTML 파싱

HTML 을 받아 원하는 요소(제목·링크 등)를 뽑아냅니다.
- `select(...)` : CSS 선택자로 **여러 개** 찾기 → 리스트
- `select_one(...)` : **하나** 찾기

> CSS 선택자: `태그`, `.클래스`, `#아이디`, `부모 자식` 형태로 지정합니다.

In [ ]:
from bs4 import BeautifulSoup

# 예시 HTML (실제로는 requests 로 받은 res.text 를 사용)
html = '''
<html>
  <body>
    <ul class="book-list">
      <li class="book"><a href="/book/1">파이썬 입문</a><span class="price">25000</span></li>
      <li class="book"><a href="/book/2">데이터 분석</a><span class="price">30000</span></li>
      <li class="book"><a href="/book/3">웹 스크래핑</a><span class="price">28000</span></li>
    </ul>
  </body>
</html>
'''

# HTML 문자열을 파싱해 탐색 가능한 객체로
soup = BeautifulSoup(html, 'html.parser')

# 'li.book' : class가 book 인 li 요소들을 모두 선택 → 리스트
items = soup.select('li.book')
print('찾은 개수:', len(items))

In [ ]:
# 각 요소에서 텍스트와 속성(href) 추출
for item in items:
    title = item.select_one('a').get_text(strip=True)   # 링크 텍스트
    link = item.select_one('a')['href']                 # href 속성값
    price = item.select_one('span.price').get_text(strip=True)
    print(f'{title} | {price}원 | {link}')

## 5. 파싱 결과 → DataFrame (전체 흐름 종합)

지금까지 배운 것을 모두 합칩니다:
**반복(02) + 딕셔너리(01) + 예외 대비(03) + DataFrame(04) + 파싱(05)**

In [ ]:
import pandas as pd

# 파싱한 요소들을 '딕셔너리들의 리스트' 로 모은다
rows = []
for item in items:
    rows.append({
        'title': item.select_one('a').get_text(strip=True),
        'price': int(item.select_one('span.price').get_text(strip=True)),  # 숫자로 변환
        'link': item.select_one('a')['href'],
    })

# 리스트 → DataFrame (04편)
df = pd.DataFrame(rows)
print(df)

# 04편에서 배운 분석 바로 적용: 가격 높은 순
print('\n가장 비싼 책:', df.sort_values('price', ascending=False).iloc[0]['title'])

# CSV 로 저장 (03편) — 한글 안깨지게 utf-8-sig
df.to_csv('parsed_books.csv', index=False, encoding='utf-8-sig')
print('CSV 저장 완료')

## 정리 — 웹스크래핑 전체 흐름
1. **requests.get(url, params=...)** 로 요청
2. **try/except + raise_for_status()** 로 안전하게
3. JSON 은 **res.json()**, HTML 은 **BeautifulSoup(res.text)**
4. **select / select_one + get_text / [속성]** 으로 추출
5. **'딕셔너리들의 리스트' → pd.DataFrame → 분석/저장**